# Fase 5: Purificação de Features e Neutralização Fatorial
### (Boruta-SHAP / CFI e Neutralização Setorial/Market Cap)

Este notebook demonstra o pipeline de purificação em dois estágios e avaliação de multicolinearidade (VIF):
1. **Neutralização Setorial (Estágio 1)**: Regressão OLS contra dummies setoriais $\rightarrow$ extração de $f_{i, \text{setorial}}$.
2. **Neutralização por Tamanho / Size Bias (Estágio 2)**: Regressão log-linear e quadrática contra Market Cap $\rightarrow$ resíduo puro $\epsilon_i$.
3. **Diagnóstico de Multicolinearidade (VIF)** antes e depois da purificação.

In [1]:
# Cell 1: Importações e Simulação de Dados de Ativos com Vieses de Setor e Tamanho
import os
import sys
import pandas as pd
import numpy as np

sys.path.insert(0, os.path.abspath('..'))

from src.features.purification import (
    neutralize_feature_two_stage,
    compute_vif_dataframe,
    FeaturePurifier,
    compute_vif,
    select_informative_features,
    neutralize_factors,
)

np.random.seed(42)
n_assets = 100

sectors = np.random.choice(['Technology', 'Financials', 'Healthcare', 'Energy'], size=n_assets)
market_caps = np.random.uniform(1e8, 5e11, size=n_assets)

# Simulação de um indicador bruto contaminado pelo Setor Tecnológico e pela dimensão (Market Cap)
tech_bias = np.where(sectors == 'Technology', 2.5, 0.0)
size_bias = np.log(market_caps) * 0.4
raw_indicator = np.random.normal(0, 1, size=n_assets) + tech_bias + size_bias

# Simulação de um segundo indicador fortemente correlacionado (gerador de multicolinearidade)
correlated_indicator = raw_indicator * 0.85 + np.random.normal(0, 0.2, size=n_assets)

df_portfolio = pd.DataFrame({
    'ticker': [f'STOCK_{i}' for i in range(n_assets)],
    'sector': sectors,
    'market_cap': market_caps,
    'momentum_raw': raw_indicator,
    'mcginley_raw': correlated_indicator
})
print(f"Dataset sintético criado com {len(df_portfolio)} ativos.")
print(df_portfolio.head(3))

In [2]:
# Cell 2: Verificação do VIF Inicial (Antes da Neutralização)
features_raw = df_portfolio[['momentum_raw', 'mcginley_raw']]
vif_before = compute_vif_dataframe(features_raw)

print("=== VARIANCE INFLATION FACTOR (VIF) ANTES DA PURIFICAÇÃO ===")
print(vif_before)

In [3]:
# Cell 3: Aplicação da Neutralização Fatorial em Duas Etapas
purifier = FeaturePurifier(df_portfolio, sector_col='sector', market_cap_col='market_cap')
purified_features = purifier.neutralize_all_features(['momentum_raw', 'mcginley_raw'])

df_combined = pd.concat([df_portfolio, purified_features], axis=1)
print("Features purificadas geradas com sucesso.")

In [4]:
# Cell 4: Verificação do VIF Após a Purificação
vif_after = compute_vif_dataframe(purified_features)

print("\n=== VARIANCE INFLATION FACTOR (VIF) APÓS A PURIFICAÇÃO ===")
print(vif_after)

print("\n=== COMPARAÇÃO DE VARIÁVEIS PARA ATIVOS SELECIONADOS ===")
print(df_combined[['ticker', 'sector', 'momentum_raw', 'momentum_raw_purified']].head(5))